## Assignment 3 - Addressing feedback from assignment 2

**Purpose:**

- It was noted how the score_pct_diff contributes 47% of the model's predictive weight and I wanted to go back into the code and examine my results, and see if there was something I missed. Also, the AUC-ROC score I achieved of .970 was noted as suspiciously high, so I would like to test tht as well.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


In [2]:
# Load the exact dataset used in Assignment 2
df = pd.read_csv("modeling_ready_dataset.csv")

display(pd.DataFrame([{
    "Rows": df.shape[0],
    "Columns": df.shape[1],
    "Seasons": f"{df['season'].min()}-{df['season'].max()}",
    "Playoff Teams": df["made_playoffs"].sum(),
    "Non-Playoff Teams": (df["made_playoffs"] == 0).sum(),
}], index=["Dataset Summary"]))
 

,Rows,Columns,Seasons,Playoff Teams,Non-Playoff Teams
Dataset Summary,256,68,2018-2025,108,148


All features used in Assignment 2:

In [3]:
all_feature_cols = [c for c in df.columns
                    if c not in ["season", "team", "made_playoffs", "exp_diff_corr"]]
 
# Same features but without score_pct_diff
no_spd_feature_cols = [c for c in all_feature_cols if c != "score_pct_diff"]
display(pd.DataFrame([{
    "WITH score_pct_diff": len(all_feature_cols),
    "WITHOUT score_pct_diff": len(no_spd_feature_cols),
}], index=["Feature Count"]))

,WITH score_pct_diff,WITHOUT score_pct_diff
Feature Count,64,63


In [4]:
target_col = "made_playoffs"

Using the same expanding window folds from assignment 2

In [5]:
folds = []
for test_year in range(2021, 2026):
    train_mask = df["season"] < test_year
    test_mask = df["season"] == test_year
 
    folds.append({
        "fold_num": test_year - 2020,
        "test_year": test_year,
        "train_seasons": sorted(df.loc[train_mask, "season"].unique()),
        "X_train_all": df.loc[train_mask, all_feature_cols].copy(),
        "X_train_no_spd": df.loc[train_mask, no_spd_feature_cols].copy(),
        "y_train": df.loc[train_mask, target_col].copy(),
        "X_test_all": df.loc[test_mask, all_feature_cols].copy(),
        "X_test_no_spd": df.loc[test_mask, no_spd_feature_cols].copy(),
        "y_test": df.loc[test_mask, target_col].copy(),
    })
 
fold_summary = pd.DataFrame([{
    "Fold": f["fold_num"],
    "Train Seasons": f"2018-{f['train_seasons'][-1]}",
    "Train N": len(f["y_train"]),
    "Test Year": f["test_year"],
    "Test N": len(f["y_test"]),
    "Train Playoff %": f"{f['y_train'].mean() * 100:.1f}%",
    "Test Playoff %": f"{f['y_test'].mean() * 100:.1f}%",
} for f in folds])

fold_summary

,Fold,Train Seasons,Train N,Test Year,Test N,Train Playoff %,Test Playoff %
0,1,2018-2020,96,2021,32,39.6%,43.8%
1,2,2018-2021,128,2022,32,40.6%,43.8%
2,3,2018-2022,160,2023,32,41.2%,43.8%
3,4,2018-2023,192,2024,32,41.7%,43.8%
4,5,2018-2024,224,2025,32,42.0%,43.8%


Using the same hyperparameter grid from assignment 2

In [6]:
param_grid_lr = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0],
    "penalty": ["l1", "l2"],
}

Running Logistic Regression with grid search across all expanding window folds.
 
For each fold:

    1. Scale features...fit on training data ONLY to prevent leakage

    2. Grid search over C and penalty using stratified 5-fold inner CV
    
    3. Evaluate best model on the held-out test season

In [9]:
def run_lr_pipeline(folds, train_key, test_key):
    results = []
    models_info = []
 
    for fold in folds:
        # Step 1: Scale features (training-set parameters only)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(fold[train_key])
        X_test_scaled = scaler.transform(fold[test_key])
 
        # Step 2: Grid search with inner CV
        cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        lr = LogisticRegression(solver="saga", max_iter=5000, random_state=42)
 
        grid_search = GridSearchCV(
            estimator=lr,
            param_grid=param_grid_lr,
            cv=cv_inner,
            scoring="f1",
            n_jobs=-1,
            refit=True,
        )
        grid_search.fit(X_train_scaled, fold["y_train"])
 
        # Step 3: Evaluate on held-out test season
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
 
        y_pred = best_model.predict(X_test_scaled)
        y_prob = best_model.predict_proba(X_test_scaled)[:, 1]
 
        results.append({
            "Fold": fold["fold_num"],
            "Test Year": fold["test_year"],
            "Accuracy": accuracy_score(fold["y_test"], y_pred),
            "Precision": precision_score(fold["y_test"], y_pred, zero_division=0),
            "Recall": recall_score(fold["y_test"], y_pred, zero_division=0),
            "F1": f1_score(fold["y_test"], y_pred, zero_division=0),
            "AUC-ROC": roc_auc_score(fold["y_test"], y_prob),
            "Best C": best_params["C"],
            "Best Penalty": best_params["penalty"],
        })
 
        models_info.append({
            "fold": fold["fold_num"],
            "model": best_model,
            "scaler": scaler,
            "feature_names": list(fold[train_key].columns),
        })
 
    return pd.DataFrame(results), models_info
 

Runnging the pipeline with score_pct_diff, like the original model

In [10]:
results_with, models_with = run_lr_pipeline(folds, "X_train_all", "X_test_all")
results_with.set_index("Fold")

,Test Year,Accuracy,Precision,Recall,F1,AUC-ROC,Best C,Best Penalty
Fold,,,,,,,,
1,2021,0.81250,0.750000,0.857143,0.800000,0.904762,1.0,l1
2,2022,0.81250,1.000000,0.571429,0.727273,0.968254,0.1,l2
3,2023,0.96875,1.000000,0.928571,0.962963,0.992063,10.0,l1
4,2024,1.00000,1.000000,1.000000,1.000000,1.000000,10.0,l1
5,2025,0.93750,0.928571,0.928571,0.928571,0.984127,10.0,l1


In [11]:
# Averages
avg_with = results_with[["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]].agg(["mean", "std"]).T
avg_with.columns = ["Mean", "Std"]
avg_with

,Mean,Std
Accuracy,0.906250,0.088388
Precision,0.935714,0.108327
Recall,0.857143,0.167515
F1,0.883761,0.115431
AUC-ROC,0.969841,0.038227


Now, running the next pipleine without score_pct_diff

In [12]:
results_without, models_without = run_lr_pipeline(folds, "X_train_no_spd", "X_test_no_spd")
results_without.set_index("Fold")

,Test Year,Accuracy,Precision,Recall,F1,AUC-ROC,Best C,Best Penalty
Fold,,,,,,,,
1,2021,0.78125,0.733333,0.785714,0.758621,0.904762,1.0,l1
2,2022,0.81250,1.000000,0.571429,0.727273,0.960317,0.1,l2
3,2023,0.96875,1.000000,0.928571,0.962963,0.992063,10.0,l1
4,2024,1.00000,1.000000,1.000000,1.000000,1.000000,10.0,l1
5,2025,0.93750,0.928571,0.928571,0.928571,0.980159,1.0,l1


In [13]:
# Averages
avg_without = results_without[["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]].agg(["mean", "std"]).T
avg_without.columns = ["Mean", "Std"]
avg_without

,Mean,Std
Accuracy,0.900000,0.097328
Precision,0.932381,0.115490
Recall,0.842857,0.170533
F1,0.875486,0.124095
AUC-ROC,0.967460,0.038104


In [15]:
# Cross-validation averages
comparison_cv = pd.DataFrame({
    "WITH score_pct_diff": results_with[["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]].mean(),
    "WITHOUT score_pct_diff": results_without[["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]].mean(),
})
comparison_cv["Change"] = comparison_cv["WITHOUT score_pct_diff"] - comparison_cv["WITH score_pct_diff"]
comparison_cv

,WITH score_pct_diff,WITHOUT score_pct_diff,Change
Accuracy,0.906250,0.900000,-0.006250
Precision,0.935714,0.932381,-0.003333
Recall,0.857143,0.842857,-0.014286
F1,0.883761,0.875486,-0.008276
AUC-ROC,0.969841,0.967460,-0.002381


In [16]:
# 2025 holdout comparison
holdout_with = results_with[results_with["Test Year"] == 2025].iloc[0]
holdout_without = results_without[results_without["Test Year"] == 2025].iloc[0]

metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]
holdout_comparison = pd.DataFrame({
    "WITH score_pct_diff": [holdout_with[m] for m in metrics],
    "WITHOUT score_pct_diff": [holdout_without[m] for m in metrics],
}, index=metrics)
holdout_comparison["Change"] = holdout_comparison["WITHOUT score_pct_diff"] - holdout_comparison["WITH score_pct_diff"]
holdout_comparison

,WITH score_pct_diff,WITHOUT score_pct_diff,Change
Accuracy,0.937500,0.937500,0.000000
Precision,0.928571,0.928571,0.000000
Recall,0.928571,0.928571,0.000000
F1,0.928571,0.928571,0.000000
AUC-ROC,0.984127,0.980159,-0.003968


In [17]:
# Per-fold AUC comparison
auc_comparison = pd.DataFrame({
    "Fold": results_with["Fold"],
    "Test Year": results_with["Test Year"],
    "AUC WITH": results_with["AUC-ROC"],
    "AUC WITHOUT": results_without["AUC-ROC"],
})
auc_comparison["Change"] = auc_comparison["AUC WITHOUT"] - auc_comparison["AUC WITH"]
 
auc_comparison.set_index("Fold")

,Test Year,AUC WITH,AUC WITHOUT,Change
Fold,,,,
1,2021,0.904762,0.904762,0.000000
2,2022,0.968254,0.960317,-0.007937
3,2023,0.992063,0.992063,0.000000
4,2024,1.000000,1.000000,0.000000
5,2025,0.984127,0.980159,-0.003968


In [23]:
for label, minfo in [("WITH score_pct_diff", models_with[-1]),
                      ("WITHOUT score_pct_diff", models_without[-1])]:
 
    model = minfo["model"]
    feature_names = minfo["feature_names"]
 
    coef_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": model.coef_[0],
        "Abs Coefficient": np.abs(model.coef_[0]),
    }).sort_values("Abs Coefficient", ascending=False).reset_index(drop=True)
 
    coef_df["Direction"] = coef_df["Coefficient"].apply(
        lambda x: "favors playoff" if x > 0.001 else ("hurts playoff" if x < -0.001 else "(eliminated)"))
 
    n_active = (coef_df["Abs Coefficient"] > 0.001).sum()
 
    print(f"Active features: {n_active} / {len(feature_names)}")
    print(f"Best params: C={model.C}, penalty={model.penalty}")
    display(coef_df.head(15)[["Feature", "Coefficient", "Direction"]])
 

Active features: 39 / 64
Best params: C=10.0, penalty=l1


,Feature,Coefficient,Direction
0,def_adv_Air,8.878413,favors playoff
1,def_adv_YAC,5.691283,favors playoff
2,def_yds_allowed,-5.125795,hurts playoff
3,def_yds_per_play,-4.511847,hurts playoff
4,def_rush_Yds,2.942794,favors playoff
5,def_plays,-2.737718,hurts playoff
6,def_pass_ANY/A,-2.407025,hurts playoff
7,def_rush_Y/A,2.074177,favors playoff
8,def_rush_Att,1.283178,favors playoff
9,def_pass_Sk,-1.171377,hurts playoff


Active features: 22 / 63
Best params: C=1.0, penalty=l1


,Feature,Coefficient,Direction
0,def_yds_allowed,-5.133752,hurts playoff
1,def_adv_Air,4.192146,favors playoff
2,def_adv_YAC,2.906095,favors playoff
3,def_rush_Yds,1.615248,favors playoff
4,def_rush_Y/A,1.198900,favors playoff
5,off_score_pct,0.967424,favors playoff
6,def_yds_per_play,-0.804935,hurts playoff
7,off_turnover_pct,-0.357094,hurts playoff
8,def_pass_ANY/A,-0.330836,hurts playoff
9,def_pass_Rate,-0.318236,hurts playoff


In [24]:
fold_5 = folds[-1]

for label, minfo, feat_key in [("WITH score_pct_diff", models_with[-1], "X_test_all"),
                                 ("WITHOUT score_pct_diff", models_without[-1], "X_test_no_spd")]:

    model = minfo["model"]
    scaler = minfo["scaler"]
    X_test_scaled = scaler.transform(fold_5[feat_key])

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    cm = confusion_matrix(fold_5["y_test"], y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n--- {label} ---")

    cm_df = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual: No Playoff", "Actual: Made Playoff"],
        columns=["Predicted: No Playoff", "Predicted: Made Playoff"]
    )
    display(cm_df)

    display(pd.DataFrame([{
        "True Positives": tp,
        "False Positives": fp,
        "True Negatives": tn,
        "False Negatives": fn,
    }], index=["Counts"]))

    # Show misclassified teams
    test_df = df[df["season"] == 2025].copy()
    test_df["Predicted"] = y_pred
    test_df["P(Playoff)"] = y_prob

    misclassified = test_df[test_df["Predicted"] != test_df["made_playoffs"]].copy()
    if len(misclassified) > 0:
        misclassified["Actual"] = misclassified["made_playoffs"].map({1: "Made playoffs", 0: "Missed playoffs"})
        misclassified["Predicted Label"] = misclassified["Predicted"].map({1: "Made playoffs", 0: "Missed playoffs"})
        display(misclassified[["team", "Actual", "Predicted Label", "P(Playoff)"]].set_index("team"))
    else:
        print("All 32 teams correctly classified!")


--- WITH score_pct_diff ---


,Predicted: No Playoff,Predicted: Made Playoff
Actual: No Playoff,17,1
Actual: Made Playoff,1,13


,True Positives,False Positives,True Negatives,False Negatives
Counts,13,1,17,1


,Actual,Predicted Label,P(Playoff)
team,,,
Detroit Lions,Missed playoffs,Made playoffs,0.510000
Carolina Panthers,Made playoffs,Missed playoffs,0.036024



--- WITHOUT score_pct_diff ---


,Predicted: No Playoff,Predicted: Made Playoff
Actual: No Playoff,17,1
Actual: Made Playoff,1,13


,True Positives,False Positives,True Negatives,False Negatives
Counts,13,1,17,1


,Actual,Predicted Label,P(Playoff)
team,,,
Detroit Lions,Missed playoffs,Made playoffs,0.656135
Carolina Panthers,Made playoffs,Missed playoffs,0.080121


In [25]:
cv_auc_with = results_with["AUC-ROC"].mean()
cv_auc_without = results_without["AUC-ROC"].mean()
ho_auc_with = holdout_with["AUC-ROC"]
ho_auc_without = holdout_without["AUC-ROC"]

summary = pd.DataFrame([
    {"Comparison": "Cross-Validation AUC", "WITH": cv_auc_with, "WITHOUT": cv_auc_without,
     "Change": cv_auc_without - cv_auc_with},
    {"Comparison": "2025 Holdout AUC", "WITH": ho_auc_with, "WITHOUT": ho_auc_without,
     "Change": ho_auc_without - ho_auc_with},
])
summary.set_index("Comparison")

,WITH,WITHOUT,Change
Comparison,,,
Cross-Validation AUC,0.969841,0.967460,-0.002381
2025 Holdout AUC,0.984127,0.980159,-0.003968


Removing `score_pct_diff` causes negligible change in model performance.
* Cross-validation AUC drops by only 0.002 and holdout AUC drops by 0.004.
* Both models misclassify the same two teams on the 2025 holdout.
 
`score_pct_diff` is computed from end-of-season offensive and defensive
scoring percentages, both available at the same time as all other features.
It is not leaking future information.

The high AUC reflects genuine separability in NFL team performance data,
not an artifact of a single dominant feature. 
  
Without `score_pct_diff`, the model selects fewer features via L1 regularization, producing a sparser model that achieves essentially identical results.

Both models exceed the performance thresholds from Assignment 1.
